In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model


In [2]:
class CosineWithWarmup(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, total_steps, warmup_steps):
        super().__init__()
        self.base_lr = float(base_lr)
        self.total_steps = int(total_steps)
        self.warmup_steps = int(warmup_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        total = tf.cast(self.total_steps, tf.float32)
        warmup = tf.cast(self.warmup_steps, tf.float32)

        # warmup: tăng tuyến tính 0 -> base_lr
        warm = tf.minimum(step / tf.maximum(warmup, 1.0), 1.0)

        # cosine sau warmup
        denom = tf.maximum(total - warmup, 1.0)
        progress = tf.clip_by_value((step - warmup) / denom, 0.0, 1.0)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * progress))

        lr = self.base_lr * (warm * cosine + (1.0 - warm) * 0.0)
        return tf.cast(lr, tf.float32)

# Định nghĩa lại layer Cast giống lúc train (hoặc tương đương)
class Cast(keras.layers.Layer):
    def call(self, inputs):
        return tf.cast(inputs, tf.float32)


In [3]:
custom_objects = {
    'Cast': Cast,
    'CosineWithWarmup': CosineWithWarmup,   # nếu có dùng
}

model = load_model(
    'best_b4/best_b4.h5',
    custom_objects=custom_objects,
    compile=False
)


In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb4 (Functional)     │ (None, 10, 10, 1792)   │    17,673,823 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1792)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       459,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cast_3 (Cast)                   │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,135,140 (69.18 MB)

 Trainable params: 14,605,157 (55.71 MB)

 Non-trainable params: 3,529,983 (13.47 MB)

In [5]:
import numpy as np

def load_and_preprocess_image(img_path, model, scale=True):
    """
    img_path: đường dẫn tới file ảnh
    model: model keras đã load
    scale: có chia 255 hay không (tùy cách bạn train)
    """
    # Lấy size từ model: (None, H, W, C)
    _, h, w, _ = model.input_shape

    img = tf.keras.utils.load_img(img_path, target_size=(h, w))
    img_array = tf.keras.utils.img_to_array(img)

    if scale:
        img_array = img_array / 255.0   # nếu lúc train bạn cũng chia 255

    img_array = np.expand_dims(img_array, axis=0)  # (1, H, W, C)
    return img_array


In [6]:
# Đặt tên class đúng thứ tự với output của model
# Ví dụ: nếu model.output_shape = (None, 3) thì có 3 lớp:
class_names = ['CBB','CBSD','CGM','CMD','Healthy']

def predict_image(img_path, model, class_names, scale=True):
    """
    Predict 1 ảnh đơn lẻ.
    """
    img_batch = load_and_preprocess_image(img_path, model, scale=scale)

    preds = model.predict(img_batch)  # shape (1, num_classes) hoặc (1, 1) cho binary

    # Trường hợp binary classification với output 1 node sigmoid
    if preds.shape[-1] == 1:
        prob = float(preds[0][0])
        pred_label = class_names[1] if prob >= 0.5 else class_names[0]
        return pred_label, prob, preds[0]

    # Trường hợp multi-class
    probs = preds[0]
    idx = int(np.argmax(probs))
    pred_label = class_names[idx]
    pred_prob = float(probs[idx])

    return pred_label, pred_prob, probs


In [12]:
img_path = "D:\\Demo dat\\1000201771.jpg"  

pred_label, pred_prob, probs = predict_image(
    img_path=img_path,
    model=model,
    class_names=class_names,
    scale=True   # hoặc False nếu lúc train KHÔNG chia 255
)

print("Predicted label:", pred_label)
print("Probability:", pred_prob)
print("All probs:", probs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step
Predicted label: Healthy
Probability: 0.4317883253097534
All probs: [0.1229445  0.14244653 0.08872335 0.21409734 0.43178833]
